# Data Normalization Pipeline
**Project:** Mining Behavioral Risk Patterns for Diabetes Prediction  
**Group 9 — CLC01**

Pipeline:
1. Load & inspect data
2. Binarize target (3 classes → 2 classes)
3. Handle BMI outliers (experiment: IQR-cap vs. removal)
4. Normalize / scale features theo từng loại
5. Discretize features cho Association Rule Mining
6. Lưu các dataset đã xử lý

## Cell 1 — Install & Import Libraries

In [ ]:
# Uncomment nếu chạy lần đầu trên Colab
# !pip install pandas numpy scikit-learn matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler

print('Libraries loaded successfully.')

## Cell 2 — Load Dataset

In [ ]:
# Nếu dùng Google Drive:
# from google.colab import drive
# drive.mount('/content/drive')
# df = pd.read_csv('/content/drive/MyDrive/your_folder/data.csv')

# Upload trực tiếp lên Colab:
# from google.colab import files
# uploaded = files.upload()  # chọn file data.csv

df = pd.read_csv('data.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

## Cell 3 — Inspect Data Types & Basic Stats

In [ ]:
print('=== Data Types ===')
print(df.dtypes)

print('\n=== Missing Values ===')
print(df.isnull().sum())

print('\n=== Target Distribution (before binarization) ===')
print(df['Diabetes_binary'].value_counts())

print('\n=== Basic Stats (continuous features) ===')
df[['BMI', 'MentHlth', 'PhysHlth']].describe()

## Cell 4 — Binarize Target Variable
- Class 0 → 0 (No Risk)
- Class 1 (Pre-diabetes) + Class 2 (Diabetes) → 1 (At Risk)

Lý do: cả pre-diabetes và diabetes đều cần cùng một can thiệp hành vi.

In [ ]:
df['Diabetes_binary'] = df['Diabetes_binary'].apply(lambda x: 0 if x == 0.0 else 1)

counts = df['Diabetes_binary'].value_counts()
pct = df['Diabetes_binary'].value_counts(normalize=True) * 100

print('=== Target Distribution (after binarization) ===')
print(pd.DataFrame({'Count': counts, 'Percentage': pct.round(2)}))

# Visualize
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(['No Risk (0)', 'At Risk (1)'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Target Distribution After Binarization')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 1000, f'{v:,}', ha='center')
plt.tight_layout()
plt.show()

## Cell 5 — BMI Outlier Detection
993 records có BMI > 60 (không hợp lý về mặt sinh lý).  
Tạo 2 phiên bản dataset để so sánh downstream:
- **Strategy A**: IQR Capping (giữ record, cap giá trị)
- **Strategy B**: Record Removal (xóa record)

In [ ]:
# Visualize BMI distribution trước khi xử lý
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['BMI'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('BMI Distribution (Original)')
axes[0].set_xlabel('BMI')
axes[0].axvline(60, color='red', linestyle='--', label='BMI=60 threshold')
axes[0].legend()

axes[1].boxplot(df['BMI'], vert=False)
axes[1].set_title('BMI Boxplot (Original)')
axes[1].set_xlabel('BMI')

plt.tight_layout()
plt.show()

print(f'Records with BMI > 60: {(df["BMI"] > 60).sum()}')
print(f'BMI stats: min={df["BMI"].min()}, max={df["BMI"].max()}, mean={df["BMI"].mean():.2f}')

## Cell 6 — Apply BMI Outlier Strategies

In [ ]:
# --- Strategy A: IQR Capping ---
df_capped = df.copy()
Q1 = df_capped['BMI'].quantile(0.25)
Q3 = df_capped['BMI'].quantile(0.75)
IQR = Q3 - Q1
upper_cap = Q3 + 1.5 * IQR
lower_cap = Q1 - 1.5 * IQR

df_capped['BMI'] = df_capped['BMI'].clip(lower=lower_cap, upper=upper_cap)
print(f'Strategy A — IQR Cap: upper={upper_cap:.2f}, lower={lower_cap:.2f}')
print(f'BMI after capping: min={df_capped["BMI"].min()}, max={df_capped["BMI"].max()}')

# --- Strategy B: Record Removal ---
df_removed = df[df['BMI'] <= 60].copy()
print(f'\nStrategy B — Record Removal: {len(df) - len(df_removed)} records removed')
print(f'Remaining records: {len(df_removed)}')

## Cell 7 — Define Feature Groups
Phân loại features để áp dụng normalization phù hợp:
- **Binary features** (0/1): không cần scale
- **Continuous features** (BMI, MentHlth, PhysHlth): MinMaxScaler → [0, 1]
- **Ordinal features** (GenHlth, Age, Education, Income): MinMaxScaler → [0, 1] (chỉ dùng cho Logistic Regression; tree-based models không cần)

> Tree-based models (XGBoost, LightGBM, RF) là scale-invariant — không cần normalize.  
> Ta tạo 2 phiên bản: `scaled` (cho LR baseline) và `unscaled` (cho ensemble).

In [ ]:
BINARY_FEATURES = [
    'HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke',
    'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
    'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex'
]

CONTINUOUS_FEATURES = ['BMI', 'MentHlth', 'PhysHlth']  # MinMaxScaler

ORDINAL_FEATURES = ['GenHlth', 'Age', 'Education', 'Income']  # MinMaxScaler

TARGET = 'Diabetes_binary'

SCALE_FEATURES = CONTINUOUS_FEATURES + ORDINAL_FEATURES  # features cần scale

print('Binary features:', BINARY_FEATURES)
print('\nContinuous features (MinMaxScaler):', CONTINUOUS_FEATURES)
print('Ordinal features (MinMaxScaler):', ORDINAL_FEATURES)

## Cell 8 — Apply MinMaxScaler
Scale continuous + ordinal features về [0, 1].  
Áp dụng trên cả 2 phiên bản dataset (capped & removed).

In [ ]:
scaler = MinMaxScaler()

def apply_scaling(dataframe, scale_cols, label):
    df_scaled = dataframe.copy()
    df_scaled[scale_cols] = scaler.fit_transform(df_scaled[scale_cols])
    print(f'[{label}] Scaled {len(scale_cols)} features → range [0, 1]')
    print(df_scaled[scale_cols].describe().loc[['min', 'max']].round(4))
    return df_scaled

# Strategy A — IQR Capped + Scaled
df_capped_scaled = apply_scaling(df_capped, SCALE_FEATURES, 'Capped + Scaled')

print()

# Strategy B — Removed + Scaled
df_removed_scaled = apply_scaling(df_removed, SCALE_FEATURES, 'Removed + Scaled')

## Cell 9 — Discretize Features for Association Rule Mining
ARM (FP-Growth) yêu cầu dữ liệu dạng categorical/binary.  
Discretize các continuous/ordinal features thành bins có ý nghĩa lâm sàng.

In [ ]:
# Dùng df_capped (chưa scale) làm base cho ARM — ARM cần giá trị gốc để bin
df_arm = df_capped.copy()

# BMI → 5 bins lâm sàng
df_arm['BMI_cat'] = pd.cut(
    df_arm['BMI'],
    bins=[0, 18.5, 25, 30, 35, float('inf')],
    labels=['Underweight', 'Normal', 'Overweight', 'Obese', 'SeverelyObese']
)

# Age → 3 nhóm
df_arm['Age_cat'] = pd.cut(
    df_arm['Age'],
    bins=[0, 4, 9, 13],
    labels=['YoungAdult_18-44', 'MiddleAged_45-64', 'Senior_65+']
)

# GenHlth → Poor / Fair / Good
df_arm['GenHlth_cat'] = df_arm['GenHlth'].map({
    1.0: 'Excellent', 2.0: 'VeryGood', 3.0: 'Good', 4.0: 'Fair', 5.0: 'Poor'
})

# MentHlth → 0 days / 1-13 days / 14-30 days
# right=False + include_lowest=True để bắt đúng giá trị 0
df_arm['MentHlth_cat'] = pd.cut(
    df_arm['MentHlth'],
    bins=[0, 1, 14, 31],
    labels=['None', 'Moderate', 'High'],
    right=False,
    include_lowest=True
)

# PhysHlth → tương tự MentHlth
df_arm['PhysHlth_cat'] = pd.cut(
    df_arm['PhysHlth'],
    bins=[0, 1, 14, 31],
    labels=['None', 'Moderate', 'High'],
    right=False,
    include_lowest=True
)

# Income → Low / Medium / High
df_arm['Income_cat'] = pd.cut(
    df_arm['Income'],
    bins=[0, 3, 6, 8],
    labels=['LowIncome', 'MidIncome', 'HighIncome']
)

# Education → Low / Medium / High
df_arm['Education_cat'] = pd.cut(
    df_arm['Education'],
    bins=[0, 2, 4, 6],
    labels=['LowEdu', 'MidEdu', 'HighEdu']
)

print('Discretized columns added:')
cat_cols = ['BMI_cat', 'Age_cat', 'GenHlth_cat', 'MentHlth_cat', 'PhysHlth_cat', 'Income_cat', 'Education_cat']
for col in cat_cols:
    print(f'  {col}: {df_arm[col].value_counts().to_dict()}')

## Cell 10 — Verify Final Datasets
Kiểm tra lại shape, range, và không có NaN trước khi lưu.

In [ ]:
datasets = {
    'df_capped_scaled': df_capped_scaled,
    'df_removed_scaled': df_removed_scaled,
    'df_arm': df_arm
}

for name, d in datasets.items():
    print(f'--- {name} ---')
    print(f'  Shape: {d.shape}')
    print(f'  Missing values: {d.isnull().sum().sum()}')
    print(f'  Target distribution: {d["Diabetes_binary"].value_counts().to_dict()}')
    print()

## Cell 11 — Save Processed Datasets
Lưu 3 file:
- `data_capped_scaled.csv` — dùng cho Logistic Regression baseline
- `data_removed_scaled.csv` — phiên bản so sánh BMI strategy
- `data_arm.csv` — dùng cho FP-Growth Association Rule Mining

In [ ]:
df_capped_scaled.to_csv('data_capped_scaled.csv', index=False)
df_removed_scaled.to_csv('data_removed_scaled.csv', index=False)
df_arm.to_csv('data_arm.csv', index=False)

print('Saved:')
print('  data_capped_scaled.csv  →  for Logistic Regression & Ensemble models (Strategy A)')
print('  data_removed_scaled.csv →  for comparison experiment (Strategy B)')
print('  data_arm.csv            →  for FP-Growth Association Rule Mining')

# Optional: download nếu đang dùng Colab
# from google.colab import files
# files.download('data_capped_scaled.csv')
# files.download('data_removed_scaled.csv')
# files.download('data_arm.csv')

## Cell 12 — Summary
Tóm tắt những gì đã làm và dataset nào dùng cho bước nào.

In [ ]:
summary = pd.DataFrame({
    'Dataset': ['data_capped_scaled.csv', 'data_removed_scaled.csv', 'data_arm.csv'],
    'BMI Strategy': ['IQR Capping', 'Record Removal', 'IQR Capping (unscaled)'],
    'Scaling': ['MinMaxScaler on BMI, MentHlth, PhysHlth, GenHlth, Age, Education, Income',
                'MinMaxScaler on BMI, MentHlth, PhysHlth, GenHlth, Age, Education, Income',
                'No scaling — original values + discretized bins'],
    'Use For': [
        'Logistic Regression baseline + XGBoost/LightGBM/RF (tree models ignore scale)',
        'BMI outlier strategy comparison experiment',
        'FP-Growth Association Rule Mining'
    ],
    'Records': [len(df_capped_scaled), len(df_removed_scaled), len(df_arm)]
})

print(summary.to_string(index=False))